# Volatility Forecasting with FinBERT News Sentiment
**Author:** Md Tanjib Mahmud
**Supervisor:** Jun.-Prof. Dr. Melina Ludolph, Otto von Guericke University Magdeburg

**Research question:** Do news-sentiment features extracted with FinBERT improve one-day-ahead volatility forecasts beyond classical GARCH(1,1) and EWMA models?

**Data sources:**
- Daily stock prices: Yahoo Finance (via the `yfinance` package)
- News headlines: FNSPID dataset (Dong et al., 2024), hosted on Hugging Face

## Step 1: Sample selection and daily price data

**What this cell does**
1. Downloads the current list of S&P 500 constituents from Wikipedia (503 tickers).
2. Looks up each company's current market capitalization on Yahoo Finance and ranks them from largest to smallest.
3. Downloads daily adjusted closing prices (corrected for splits and dividends) for the 80 largest companies, 1 January 2010 to 31 December 2023.
4. Keeps only stocks with complete price history (data on at least 99% of trading days), then selects the 50 largest of these.

**Why**
- Large companies are liquid and receive much more news coverage, which the sentiment analysis needs.
- Requiring complete history ensures all models are estimated on the same time period for every stock.

**Results**
- 10 stocks were dropped because they were not yet listed for the whole period: META, TSLA, ABBV, PLTR, PANW, ANET, CRWD, DELL, GEV, UBER.
- Final dataset: 3,522 trading days × 50 stocks (4 January 2010 to 29 December 2023), with no missing values.
- Saved as `prices_top50_daily.csv`.

**Limitations and open points**
- **Survivorship bias:** ranking by today's market cap selects companies that grew successfully, which may understate the volatility of the broader market.
- The final samples used in the analysis are selected in Step 3 based on news coverage; the 80 largest companies serve as the starting point for the news extraction. The 50-stock price file saved here is not used further.

In [ ]:
!pip install -q yfinance

import pandas as pd, requests, io, yfinance as yf

# 1. Get the current S&P 500 list from Wikipedia
html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
                    headers={"User-Agent": "Mozilla/5.0"}).text
sp500 = pd.read_html(io.StringIO(html))[0]
tickers = sp500["Symbol"].str.replace(".", "-", regex=False).tolist()  # BRK.B -> BRK-B for Yahoo
print(len(tickers), "S&P 500 tickers")

# 2. Get each company's current market cap
caps = {}
for t in tickers:
    try:
        caps[t] = yf.Ticker(t).fast_info.market_cap
    except Exception:
        pass
caps = pd.Series(caps).dropna().sort_values(ascending=False)

# 3. Download daily adjusted prices for the 80 largest (spares, in case some lack full history)
START, END = "2010-01-01", "2024-01-01"
prices = yf.download(caps.index[:80].tolist(), start=START, end=END,
                     auto_adjust=True, progress=False)["Close"]

# 4. Keep only stocks with complete history, then the 50 largest of those
complete = prices.columns[prices.notna().mean() > 0.99]
top50 = [t for t in caps.index if t in complete][:50]
prices = prices[top50]

print("Dropped (incomplete history):", [t for t in caps.index[:80] if t not in complete])
print("Final 50:", top50)
print(prices.shape, prices.index.min().date(), prices.index.max().date())

prices.to_csv("prices_top50_daily.csv")

503 S&P 500 tickers


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['GEV']: YFPricesMissingError('possibly delisted; no price data found  (1d 2010-01-01 -> 2024-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1262322000, endDate = 1704085200")')


Dropped (incomplete history): ['META', 'TSLA', 'ABBV', 'PLTR', 'PANW', 'ANET', 'CRWD', 'DELL', 'GEV', 'UBER']
Final 50: ['AAPL', 'NVDA', 'MSFT', 'GOOGL', 'GOOG', 'AMZN', 'AVGO', 'LLY', 'BRK-B', 'WMT', 'JPM', 'AMD', 'MU', 'V', 'JNJ', 'XOM', 'MA', 'COST', 'KO', 'CVX', 'PG', 'CSCO', 'ORCL', 'MRK', 'UNH', 'HD', 'BAC', 'GE', 'INTC', 'PM', 'AMAT', 'MS', 'LRCX', 'CAT', 'TXN', 'TMO', 'RTX', 'NFLX', 'GS', 'WFC', 'AMGN', 'LIN', 'IBM', 'AXP', 'GILD', 'DIS', 'C', 'APH', 'MCD', 'KLAC']
(3522, 50) 2010-01-04 2023-12-29


## Step 2a: Inspect the FNSPID news dataset

**What this cell does**
Lists the files in the FNSPID dataset on Hugging Face and their sizes, without downloading them.

**Why**
The dataset is very large. Checking its structure first avoids downloading unnecessary data and running out of memory in Colab.

**Result**
- `Stock_news/All_external.csv`: 5.73 GB
- `Stock_news/nasdaq_exteral_data.csv`: 23.23 GB
- `Stock_price/full_history.zip`: 0.59 GB (not used; prices come from Yahoo Finance)

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import HfApi

api = HfApi()
for f in api.list_repo_tree("Zihan1004/FNSPID", repo_type="dataset", recursive=True):
    size = getattr(f, "size", None)
    print(f.path, f"{size/1e9:.2f} GB" if size else "")

Stock_news 
Stock_price 
README.md 0.00 GB
Stock_news/All_external.csv 5.73 GB
Stock_news/nasdaq_exteral_data.csv 23.23 GB
Stock_price/full_history.zip 0.59 GB


## Step 2b: Preview the news files

**What this cell does**
Reads only the first 5 rows of each news file to inspect the columns and content.

**Result**
- Both files contain the columns needed for this thesis: `Date`, `Article_title` (headline) and `Stock_symbol` (ticker).
- The large size of the Nasdaq file comes from full article texts and four automatic summaries, which are not used here.

**Issues identified**
- Some headlines are market-wide roundups tagged to individual stocks (e.g. "Stocks That Hit 52-Week Highs On Friday"). They carry no company-specific information and will be filtered out.
- Headlines carry no reliable time of day: in the Nasdaq file used for the analysis, 99.8% of timestamps are exactly 00:00 UTC (the preview of All_external.csv shows some real times, but these are not used). A headline dated day d is therefore used only for forecasts of the next trading day after d, so all news is public before that trading day starts.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 80, "display.width", 200)

for name in ["All_external.csv", "nasdaq_exteral_data.csv"]:
    path = f"hf://datasets/Zihan1004/FNSPID/Stock_news/{name}"
    df = pd.read_csv(path, nrows=5)
    print("=====", name)
    print(list(df.columns))
    print(df.head(3).T)

===== All_external.csv
['Date', 'Article_title', 'Stock_symbol', 'Url', 'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary']
                                                                                                0                                                                                1  \
Date                                                                      2020-06-05 06:30:54 UTC                                                          2020-06-03 06:45:20 UTC   
Article_title                                             Stocks That Hit 52-Week Highs On Friday                                       Stocks That Hit 52-Week Highs On Wednesday   
Stock_symbol                                                                                    A                                                                                A   
Url               https://www.benzinga.com/news/20/06/16190091/stocks-that-hit-52-week-highs-o..

## Step 2c: Extract headlines from `All_external.csv`

**What this cell does**
Reads the 5.73 GB file in chunks of 500,000 rows, keeps only the date, headline and ticker columns, and keeps only headlines for the 80 candidate companies. Reading in chunks prevents Colab from running out of memory.

**Results**
- 122,648 headlines kept, from 22 June 2009 to 11 June 2020.
- Coverage is uneven: only 62 of the 80 candidate companies appear in this file. Among the 50 stocks with complete price history from Step 1, 9 have no headlines at all (MSFT, BRK-B, COST, CVX, GE, PM, RTX, GS, LIN), and AAPL and AMZN appear only in 2020.
- About 3.8% of headlines are same-day roundups shared by five or more of the tickers, and about 5,000 headline–ticker pairs are duplicates.
- Saved as `news_allexternal_filtered.csv`.

**Conclusion**
This file alone does not cover the sample well enough and ends in mid-2020. The Nasdaq file is therefore also needed.

In [ ]:
import pandas as pd

TICKERS = set(caps.index[:80])          # the 80 largest companies from step 1
url = "hf://datasets/Zihan1004/FNSPID/Stock_news/All_external.csv"

keep, n_rows = [], 0
for i, chunk in enumerate(pd.read_csv(url, usecols=["Date", "Article_title", "Stock_symbol"],
                                      chunksize=500_000, dtype=str)):
    n_rows += len(chunk)
    keep.append(chunk[chunk["Stock_symbol"].isin(TICKERS)])
    if i % 5 == 0:
        print(f"{n_rows:,} rows read, {sum(len(k) for k in keep):,} kept")

news = pd.concat(keep, ignore_index=True)
news["Date"] = pd.to_datetime(news["Date"], utc=True, errors="coerce")
news = news.dropna(subset=["Date", "Article_title"])

print("\nTotal kept:", f"{len(news):,}", "| range:", news.Date.min().date(), "to", news.Date.max().date())
print("\nHeadlines per year:\n", news.groupby(news.Date.dt.year).size())
print("\nTop 15 tickers:\n", news.Stock_symbol.value_counts().head(15))

# How many headlines are shared by 5+ of our tickers on the same day (roundups)?
day = news.Date.dt.date
shared = news.groupby([day, "Article_title"])["Stock_symbol"].transform("nunique")
print("\nShare of headlines that are same-day roundups (5+ tickers):", round((shared >= 5).mean(), 3))

news.to_csv("news_allexternal_filtered.csv", index=False)

500,000 rows read, 22,642 kept
3,000,000 rows read, 104,974 kept
5,500,000 rows read, 122,648 kept
8,000,000 rows read, 122,648 kept
10,500,000 rows read, 122,648 kept
13,000,000 rows read, 122,648 kept

Total kept: 122,648 | range: 2009-06-22 to 2020-06-11

Headlines per year:
 Date
2009      446
2010     3944
2011     5794
2012     6612
2013     6539
2014     7243
2015     9256
2016    11947
2017    12940
2018    16825
2019    24932
2020    16170
dtype: int64

Top 15 tickers:
 Stock_symbol
GILD    4974
HD      4453
AVGO    4149
BLK     3989
TXN     3872
TJX     3817
AXP     3783
PANW    3591
LLY     3538
MRK     3335
MS      3246
MU      3149
NVDA    3146
NFLX    3028
VZ      2972
Name: count, dtype: int64

Share of headlines that are same-day roundups (5+ tickers): 0.038


## Step 2d: Extract headlines from `nasdaq_exteral_data.csv`

**What this cell does**
Same procedure as Step 2c, applied to the 23.23 GB Nasdaq file, which extends to 2023. The result is saved in compressed form as `news_nasdaq_filtered.csv.gz`.

**Results**
- 409,963 headlines kept (2009 to January 2024); about 400,000 unique after removing duplicates.
- Every headline from `All_external.csv` also appears in this file, so only the Nasdaq file is used from here on.
- Coverage is uneven and has gaps caused by how the dataset was collected. For example, AAPL has headlines only in 2020, 2022 and 2023, and many stocks have none in 2021. These gaps are treated as missing data, not as days without news.
- Coverage is most complete between January 2017 and May 2020, when 47 of the candidate stocks have at least 10 headlines in at least 90% of months.

In [ ]:
import pandas as pd

TICKERS = set(caps.index[:80])
url = "hf://datasets/Zihan1004/FNSPID/Stock_news/nasdaq_exteral_data.csv"

keep, n_rows = [], 0
for i, chunk in enumerate(pd.read_csv(url, usecols=["Date", "Article_title", "Stock_symbol"],
                                      chunksize=500_000, dtype=str)):
    n_rows += len(chunk)
    keep.append(chunk[chunk["Stock_symbol"].isin(TICKERS)])
    if i % 5 == 0:
        print(f"{n_rows:,} rows read, {sum(len(k) for k in keep):,} kept")

nasdaq = pd.concat(keep, ignore_index=True)
nasdaq["Date"] = pd.to_datetime(nasdaq["Date"], utc=True, errors="coerce")
nasdaq = nasdaq.dropna(subset=["Date", "Article_title"])

print("\nTotal kept:", f"{len(nasdaq):,}", "| range:", nasdaq.Date.min().date(), "to", nasdaq.Date.max().date())
print("\nHeadlines per year:\n", nasdaq.groupby(nasdaq.Date.dt.year).size())

nasdaq.to_csv("news_nasdaq_filtered.csv.gz", index=False)   # compressed, so it's small enough to send

500,000 rows read, 60,611 kept
3,000,000 rows read, 309,957 kept
5,500,000 rows read, 392,289 kept
8,000,000 rows read, 409,963 kept
10,500,000 rows read, 409,963 kept
13,000,000 rows read, 409,963 kept
15,500,000 rows read, 409,963 kept

Total kept: 409,963 | range: 2009-06-22 to 2024-01-09

Headlines per year:
 Date
2009      484
2010     4830
2011     7327
2012    10112
2013    11692
2014    15541
2015    20276
2016    27415
2017    35759
2018    39560
2019    44398
2020    42823
2021    26692
2022    48571
2023    74413
2024       70
dtype: int64


## Step 3: Sample selection based on news coverage

**What this cell does**
1. Removes duplicate headlines (same stock, same day, same title).
2. Counts headlines per stock per month (New York time).
3. Defines a stock as *covered* in a month if it has at least 10 headlines, and selects stocks covered in at least 90% of the months of each window:
   - Main sample: January 2017 – May 2020 (includes the COVID-19 crash)
   - Robustness sample: January 2022 – December 2023 (includes the 2022 bear market)
4. Downloads daily adjusted prices for all selected stocks from 2010, so GARCH models can be estimated on several years of history before each window.

*Note:* Headline dates carry no time of day (00:00 UTC), so converting to New York time moves a headline to the previous calendar day; this affects only the monthly counts used for stock selection, not the daily news features.


**Why**
FNSPID's news coverage has gaps caused by how the data was collected (e.g. no Apple headlines in 2017–2019). Testing the news-based model only where coverage is continuous prevents these gaps from being mistaken for days without news.

**Limitation**
Some of the largest companies (e.g. AAPL, MSFT, AMZN, GOOGL) are not included in the main sample because FNSPID lacks their news in that period.

In [ ]:
import pandas as pd, yfinance as yf

# 1. Load the Nasdaq headlines and remove duplicates
news = pd.read_csv("news_nasdaq_filtered.csv.gz", parse_dates=["Date"])
news["time_ny"] = news["Date"].dt.tz_convert("America/New_York")
key = (news["Stock_symbol"] + "|" + news["time_ny"].dt.date.astype(str) + "|"
       + news["Article_title"].str.lower().str.strip())
news = news[~key.duplicated()].copy()
print("Unique headlines:", f"{len(news):,}")

# 2. Headlines per stock per month
news["month"] = news["time_ny"].dt.tz_localize(None).dt.to_period("M")
monthly = news.groupby(["Stock_symbol", "month"]).size().unstack(fill_value=0)

def covered_stocks(start, end, min_headlines=10, min_share=0.9):
    months = pd.period_range(start, end, freq="M")
    window = monthly.reindex(columns=months, fill_value=0)
    share_covered = (window >= min_headlines).mean(axis=1)
    return sorted(share_covered[share_covered >= min_share].index)

main_stocks = covered_stocks("2017-01", "2020-05")
robust_stocks = covered_stocks("2022-01", "2023-12")
print(f"\nMain sample ({len(main_stocks)}):", main_stocks)
print(f"\nRobustness sample ({len(robust_stocks)}):", robust_stocks)

# 3. Daily prices for every stock in either sample
all_stocks = sorted(set(main_stocks) | set(robust_stocks))
prices = yf.download(all_stocks, start="2010-01-01", end="2024-01-01",
                     auto_adjust=True, progress=False)["Close"]
print("\nPrices:", prices.shape)
print("Stocks with the shortest history (first price date):")
print(prices.apply(lambda s: s.first_valid_index()).sort_values().tail(5))

# 4. Save everything
prices.to_csv("prices_sample_daily.csv")
news.drop(columns=["month"]).to_csv("news_clean.csv.gz", index=False)
pd.Series(main_stocks, name="ticker").to_csv("sample_main.csv", index=False)
pd.Series(robust_stocks, name="ticker").to_csv("sample_robust.csv", index=False)

Unique headlines: 401,696

Main sample (47): ['ABBV', 'ABT', 'ACN', 'ADI', 'AMAT', 'AMD', 'AMGN', 'ANET', 'APH', 'AVGO', 'AXP', 'BAC', 'BLK', 'C', 'CAT', 'COP', 'COST', 'CRM', 'CVX', 'DE', 'DHR', 'GE', 'GILD', 'GS', 'HD', 'JNJ', 'KO', 'MRK', 'MS', 'MU', 'NEE', 'NFLX', 'NVDA', 'ORCL', 'PANW', 'PEP', 'PFE', 'PM', 'QCOM', 'T', 'TJX', 'TMO', 'TMUS', 'TXN', 'UNP', 'V', 'WFC']

Robustness sample (44): ['ABBV', 'ABT', 'ACN', 'ADI', 'AMAT', 'AMD', 'AMGN', 'ANET', 'CAT', 'COP', 'COST', 'CRM', 'CRWD', 'CVX', 'DE', 'DELL', 'DHR', 'DIS', 'GE', 'GILD', 'GOOG', 'GS', 'INTC', 'KO', 'MRK', 'MS', 'MU', 'NEE', 'NVDA', 'ORCL', 'PANW', 'PEP', 'PM', 'QCOM', 'T', 'TJX', 'TMO', 'TMUS', 'TXN', 'UNP', 'V', 'WFC', 'WMT', 'XOM']

Prices: (3522, 54)
Stocks with the shortest history (first price date):
Ticker
PANW   2012-07-20
ABBV   2013-01-02
ANET   2014-06-06
DELL   2016-08-17
CRWD   2019-06-12
dtype: datetime64[ns]


## Step 4: Daily returns, data corrections and descriptive statistics

**What this cell does**
1. Computes daily log returns in percent: r(t) = 100 × ln(P(t) / P(t−1)). Percent units help the GARCH optimizer converge.
2. Corrects three data problems in the Yahoo Finance prices:
   - DHR: the return on 5 July 2016 is removed (price adjustment error around the Fortive spin-off).
   - TMUS: returns before May 2013 are removed (the price history before the merger belongs to MetroPCS).
   - DELL: returns before 2019 are removed (the history before Dell's relisting in December 2018 belongs to a tracking stock).
3. Computes descriptive statistics of daily returns for the main sample (January 2017 – May 2020).

**Why**
Errors in the price history distort volatility estimates, because GARCH reacts strongly to extreme returns.

**Results**
- 857 trading days per stock in the main window, no missing values.
- Average annualized volatility across the 47 stocks: 32.3% (lowest: KO 21.0%; highest: AMD 60.3%).
- Returns show the usual stylized facts: average excess kurtosis 14.3 (fat tails) and average skewness −0.60 (large drops are more common than large gains), driven mainly by the COVID-19 crash in March 2020.
- Saved as `returns_daily.csv` and `descriptive_stats_main.csv`.



In [ ]:
import pandas as pd, numpy as np

prices = pd.read_csv("prices_sample_daily.csv", index_col=0, parse_dates=True)
returns = 100 * np.log(prices).diff()          # daily log returns in percent

# Data corrections
returns.loc["2016-07-05", "DHR"] = np.nan      # Fortive spin-off adjustment error
returns.loc[:"2013-04-30", "TMUS"] = np.nan    # MetroPCS before the T-Mobile merger
returns.loc[:"2018-12-31", "DELL"] = np.nan    # tracking stock before Dell's relisting

# Descriptive statistics for the main sample
main = pd.read_csv("sample_main.csv")["ticker"].tolist()
r_main = returns.loc["2017-01-01":"2020-05-31", main]
stats = pd.DataFrame({
    "Mean (%)":           r_main.mean(),
    "Std. dev. (%)":      r_main.std(),
    "Ann. volatility (%)": r_main.std() * np.sqrt(252),
    "Min (%)":            r_main.min(),
    "Max (%)":            r_main.max(),
    "Skewness":           r_main.skew(),
    "Excess kurtosis":    r_main.kurt(),
    "Observations":       r_main.count(),
})
pd.set_option("display.width", 200)
print(stats.round(2))
print("\nAverage across the 47 stocks:\n", stats.mean().round(2))

returns.to_csv("returns_daily.csv")
stats.round(4).to_csv("descriptive_stats_main.csv")

      Mean (%)  Std. dev. (%)  Ann. volatility (%)  Min (%)  Max (%)  Skewness  Excess kurtosis  Observations
ABBV      0.06           1.93                30.72   -17.74    12.90     -1.36            17.22           857
ABT       0.11           1.60                25.48   -10.30    10.38     -0.12             8.54           857
ACN       0.07           1.63                25.82    -8.76    12.10      0.15            10.11           857
ADI       0.06           2.18                34.57   -18.17    13.71     -0.58            12.04           857
AMAT      0.07           2.73                43.40   -22.76    12.06     -0.86             8.86           857
AMD       0.18           3.80                60.27   -27.75    18.19     -0.40             6.02           857
AMGN      0.06           1.64                25.98    -8.62    10.34      0.29             6.54           857
ANET      0.10           2.98                47.25   -27.75    17.76     -1.24            16.01           857
APH       

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
folder = "/content/drive/MyDrive/Thesis_new"
os.makedirs(folder, exist_ok=True)
for f in ["prices_top50_daily.csv", "news_allexternal_filtered.csv", "news_nasdaq_filtered.csv.gz",
          "prices_sample_daily.csv", "news_clean.csv.gz", "sample_main.csv", "sample_robust.csv",
          "returns_daily.csv", "descriptive_stats_main.csv"]:
    if os.path.exists(f):
        shutil.copy(f, folder)
        print("saved", f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved prices_top50_daily.csv
saved news_allexternal_filtered.csv
saved news_nasdaq_filtered.csv.gz
saved prices_sample_daily.csv
saved news_clean.csv.gz
saved sample_main.csv
saved sample_robust.csv
saved returns_daily.csv
saved descriptive_stats_main.csv


## Step 5: FinBERT sentiment scoring

**What this cell does**
1. Loads the cleaned headlines and keeps those for the sample stocks within the two windows (plus one month before each window, needed for lagged features).
2. Flags *roundup* headlines: the same headline attached to 3 or more of the sample stocks on the same day (e.g. "After Hours Most Active for May 29, 2018: BAC, GE, CSCO, …"). They are scored but can be excluded later.
3. Classifies each unique headline with FinBERT (ProsusAI/finbert; Araci, 2019), a BERT model fine-tuned on financial text, into positive, negative and neutral probabilities.
4. Defines the sentiment score of a headline as P(positive) − P(negative), ranging from −1 to +1.

**Why**
FinBERT understands financial language better than general-purpose models or word lists (e.g. "beats estimates" is positive, "cuts guidance" is negative). Using probabilities instead of only the top label keeps information about how confident the model is.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.chdir("/content/drive/MyDrive/Thesis_new")

import pandas as pd, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU!")

# 1. Headlines for the sample stocks and windows
news = pd.read_csv("news_clean.csv.gz", parse_dates=["Date"])
news["time_ny"] = news["Date"].dt.tz_convert("America/New_York")
main = pd.read_csv("sample_main.csv")["ticker"].tolist()
robust = pd.read_csv("sample_robust.csv")["ticker"].tolist()
t = news["time_ny"].dt.tz_localize(None)
in_main = news["Stock_symbol"].isin(main) & (t >= "2016-12-01") & (t < "2020-06-01")
in_robust = news["Stock_symbol"].isin(robust) & (t >= "2021-12-01") & (t < "2024-01-01")
news = news[in_main | in_robust].copy()
print("Headlines to score:", f"{len(news):,}")

# 2. Flag roundups
day = news["time_ny"].dt.date
news["n_stocks_same_headline"] = news.groupby([day, "Article_title"])["Stock_symbol"].transform("nunique")
news["roundup"] = news["n_stocks_same_headline"] >= 3
print("Roundup headlines:", f"{news['roundup'].sum():,}")

# 3. FinBERT on each unique headline
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to("cuda").half().eval()
labels = model.config.id2label
print("FinBERT label order:", labels)

titles = news["Article_title"].drop_duplicates().tolist()
probs, B = [], 256
with torch.no_grad():
    for i in range(0, len(titles), B):
        batch = tokenizer(titles[i:i+B], padding=True, truncation=True, max_length=64,
                          return_tensors="pt").to("cuda")
        probs.append(torch.softmax(model(**batch).logits.float(), dim=-1).cpu().numpy())
        if (i // B) % 100 == 0:
            print(f"{i:,} / {len(titles):,} headlines scored")
probs = np.vstack(probs)

scores = pd.DataFrame(probs, columns=[labels[i] for i in range(probs.shape[1])])
scores["Article_title"] = titles
news = news.merge(scores, on="Article_title", how="left")

# 4. Sentiment score and sanity checks
news["sentiment"] = news["positive"] - news["negative"]
news["label"] = news[["positive", "negative", "neutral"]].idxmax(axis=1)
print("\nLabel shares:\n", news["label"].value_counts(normalize=True).round(3))
pd.set_option("display.max_colwidth", 110, "display.width", 200)
print("\nMost positive:\n", news.nlargest(5, "sentiment")[["Stock_symbol", "Article_title", "sentiment"]])
print("\nMost negative:\n", news.nsmallest(5, "sentiment")[["Stock_symbol", "Article_title", "sentiment"]])

news.drop(columns=["time_ny"]).to_csv("news_scored.csv.gz", index=False)
print("\nSaved news_scored.csv.gz to Google Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
Headlines to score: 202,812
Roundup headlines: 18,765


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT label order: {0: 'positive', 1: 'negative', 2: 'neutral'}
0 / 151,226 headlines scored
25,600 / 151,226 headlines scored
51,200 / 151,226 headlines scored
76,800 / 151,226 headlines scored
102,400 / 151,226 headlines scored
128,000 / 151,226 headlines scored

Label shares:
 label
neutral     0.642
positive    0.210
negative    0.149
Name: proportion, dtype: float64

Most positive:
        Stock_symbol                                                  Article_title  sentiment
23053          ANET       Semtech (SMTC) Strengthens Clientele With LoRa Selection   0.946423
36605          COST        Will Tiffany (TIF) Manage to See Higher Earnings in Q1?   0.944936
95845            MU    Broadcom (AVGO) Strengthens Ties With HPE in Storage Server   0.944862
95096            MU   Better Pricing, New Product Release Could Boost Micron Stock   0.944825
120258         QCOM  Juniper (JNPR) Boosts Raizen's Network Competencies in Brazil   0.944246

Most negative:
        Stock_symbol       

## Step 6: Relevance filter and daily news features

**What this cell does**
1. Marks a headline as *relevant* if it names the company it is tagged to, by ticker (e.g. "(NVDA)") or company name (e.g. "Nvidia"). The list of company names was compiled by the author.
2. Assigns each headline dated day d to the next trading day after d. Features for trading day t therefore only use news published before t, which avoids look-ahead bias.
3. Computes daily features per stock, from relevant non-roundup headlines:
   - `n_news`: number of headlines (news intensity)
   - `sent`: average FinBERT sentiment, P(positive) − P(negative)
   - `neg_share`: share of headlines classified as negative
   The same features are also computed from all headlines (suffix `_all`) for a robustness check.
4. Days without a relevant headline get n_news = 0, sent = 0 and neg_share = 0.

**Why**
FNSPID tags articles to every company mentioned in the text, so only about 40% of headlines are actually about the tagged company (e.g. "Delta (DAL) Posts Rise in January Traffic" tagged to Union Pacific). Off-topic headlines would add noise to the sentiment measure.

In [ ]:
import os, re, pandas as pd, numpy as np
os.chdir("/content/drive/MyDrive/Thesis_new")

news = pd.read_csv("news_scored.csv.gz", parse_dates=["Date"])
returns = pd.read_csv("returns_daily.csv", index_col=0, parse_dates=True)
main = pd.read_csv("sample_main.csv")["ticker"].tolist()
robust = pd.read_csv("sample_robust.csv")["ticker"].tolist()

# 1. Relevance: does the headline name the company it is tagged to?
NAMES = {
    "ABBV": ["AbbVie"], "ABT": ["Abbott"], "ACN": ["Accenture"], "ADI": ["Analog Devices"],
    "AMAT": ["Applied Materials"], "AMD": ["AMD", "Advanced Micro"], "AMGN": ["Amgen"],
    "ANET": ["Arista"], "APH": ["Amphenol"], "AVGO": ["Broadcom"], "AXP": ["American Express", "Amex"],
    "BAC": ["Bank of America", "BofA"], "BLK": ["BlackRock"], "C": ["Citigroup", "Citi"],
    "CAT": ["Caterpillar"], "COP": ["ConocoPhillips", "Conoco"], "COST": ["Costco"],
    "CRM": ["Salesforce"], "CVX": ["Chevron"], "DE": ["Deere"], "DHR": ["Danaher"],
    "GE": ["General Electric", "GE"], "GILD": ["Gilead"], "GS": ["Goldman"], "HD": ["Home Depot"],
    "JNJ": ["Johnson & Johnson", "Johnson and Johnson", "J&J"], "KO": ["Coca-Cola", "Coca Cola", "Coke"],
    "MRK": ["Merck"], "MS": ["Morgan Stanley"], "MU": ["Micron"], "NEE": ["NextEra"],
    "NFLX": ["Netflix"], "NVDA": ["Nvidia"], "ORCL": ["Oracle"], "PANW": ["Palo Alto Networks"],
    "PEP": ["PepsiCo", "Pepsi"], "PFE": ["Pfizer"], "PM": ["Philip Morris"], "QCOM": ["Qualcomm"],
    "T": ["AT&T"], "TJX": ["TJX", "T.J. Maxx", "TJ Maxx"], "TMO": ["Thermo Fisher"],
    "TMUS": ["T-Mobile"], "TXN": ["Texas Instruments"], "UNP": ["Union Pacific"], "V": ["Visa"],
    "WFC": ["Wells Fargo"], "CRWD": ["CrowdStrike"], "DELL": ["Dell"], "DIS": ["Disney"],
    "GOOG": ["Alphabet", "Google"], "INTC": ["Intel"], "WMT": ["Walmart", "Wal-Mart"],
    "XOM": ["Exxon", "ExxonMobil"],
}

def is_relevant(title, ticker):
    if re.search(r"\(" + re.escape(ticker) + r"\)", title):                        # "(NVDA)"
        return True
    if len(ticker) >= 3 and re.search(r"\b" + re.escape(ticker) + r"\b", title):   # "NVDA"
        return True
    for name in NAMES[ticker]:
        flags = 0 if name.isupper() else re.IGNORECASE
        if re.search(r"(?<![A-Za-z])" + re.escape(name) + r"(?![A-Za-z])", title, flags):
            return True
    return False

news["relevant"] = [is_relevant(t, s) for t, s in zip(news["Article_title"], news["Stock_symbol"])]
print("Share of relevant headlines:", round(news["relevant"].mean(), 3))

# 2. Assign each headline to the next trading day after its publication date
trading_days = returns.index
news["pub_date"] = news["Date"].dt.tz_convert(None).dt.normalize()
pos = trading_days.searchsorted(news["pub_date"].values, side="right")
ok = pos < len(trading_days)
news = news[ok].copy()
news["date"] = trading_days[pos[ok]]

# 3. Daily features per stock
def daily_features(df, suffix=""):
    df = df.assign(is_neg=(df["label"] == "negative"))
    g = df.groupby(["Stock_symbol", "date"])
    return pd.DataFrame({f"n_news{suffix}": g.size(),
                         f"sent{suffix}": g["sentiment"].mean(),
                         f"neg_share{suffix}": g["is_neg"].mean()})

clean = news[news["relevant"] & ~news["roundup"]]
feat = daily_features(clean).join(daily_features(news, "_all"), how="outer")
feat.index.names = ["ticker", "date"]

# 4. Full panel: every stock x every trading day in its window (plus a short lead-in for lags)
panels = []
for name, stocks, start, lead_in, end in [("main", main, "2017-01-01", "2016-12-15", "2020-05-31"),
                                          ("robust", robust, "2022-01-01", "2021-12-15", "2023-12-31")]:
    days = trading_days[(trading_days >= lead_in) & (trading_days <= end)]
    idx = pd.MultiIndex.from_product([stocks, days], names=["ticker", "date"])
    p = feat.reindex(idx).fillna(0)
    p["sample"] = name
    p["in_window"] = p.index.get_level_values("date") >= pd.Timestamp(start)
    panels.append(p)
features = pd.concat(panels).reset_index()

# 5. Checks
w = features[features["in_window"]]
print("\nAverage daily features:\n", w.groupby("sample")[["n_news", "sent", "neg_share", "n_news_all", "sent_all"]].mean().round(3))
print("\nShare of stock-days with at least one relevant headline:\n", (w["n_news"] > 0).groupby(w["sample"]).mean().round(3))

r2 = (returns ** 2).stack().rename("r2").rename_axis(["date", "ticker"]).reset_index()
w = w.merge(r2, on=["date", "ticker"], how="left")
group = np.select([w["n_news"] == 0, w["neg_share"] >= 0.5], ["no relevant news", "mostly negative news"], "other news")
print("\nAverage squared return (%²) on the day after news, descriptive only:\n", w.groupby(group)["r2"].agg(["mean", "count"]).round(2))

pd.set_option("display.max_colwidth", 100)
print("\nRandom examples, relevant:\n", news[news["relevant"]].sample(5, random_state=1)[["Stock_symbol", "Article_title"]])
print("\nRandom examples, not relevant:\n", news[~news["relevant"]].sample(5, random_state=1)[["Stock_symbol", "Article_title"]])

features.to_csv("features_daily.csv.gz", index=False)
print("\nSaved features_daily.csv.gz:", features.shape)

Share of relevant headlines: 0.404

Average daily features:
         n_news   sent  neg_share  n_news_all  sent_all
sample                                                
main     1.088  0.062      0.061       2.752     0.073
robust   1.514  0.042      0.067       3.966     0.066

Share of stock-days with at least one relevant headline:
 sample
main      0.51
robust    0.66
Name: n_news, dtype: float64

Average squared return (%²) on the day after news, descriptive only:
                       mean  count
mostly negative news  5.52   4148
no relevant news      3.59  27224
other news            4.88  30951

Random examples, relevant:
        Stock_symbol                                                                           Article_title
20868          AMGN                            CVS to cover migraine drugs from Teva, Lilly; excludes Amgen
156730         AMAT  Morgan Stanley Maintains Equal-Weight on Applied Materials, Lowers Price Target to $50
144822          WFC          Finan

## Step 7: Baseline volatility forecasts

**What this cell does**
1. Produces one-day-ahead variance forecasts for every stock with four standard models:
   - GARCH(1,1) with normal errors (Bollerslev, 1986)
   - GARCH(1,1) with Student-t errors (fat tails)
   - GJR-GARCH(1,1) with Student-t errors (Glosten, Jagannathan and Runkle, 1993): negative returns can raise volatility more than positive returns (leverage effect)
   - EWMA with decay factor λ = 0.94 (J.P. Morgan RiskMetrics, 1996)
2. GARCH-type models are re-estimated at the start of every month using only returns before that month (expanding window from 2010, or from listing). The estimated parameters are then used to forecast each day of that month. The forecast for day t only uses returns up to day t−1.
3. Evaluates forecasts against the squared return of the forecast day, r(t)², a noisy but unbiased proxy for the true variance, using two loss functions:
   - MSE = mean of (r² − σ̂²)²
   - QLIKE = mean of [log σ̂² + r² / σ̂²], which is robust to noise in the volatility proxy (Patton, 2011)
4. Evaluation period: January 2018 – May 2020 (main sample) and January – December 2023 (robustness sample). The first year of each window (2017 and 2022) is reserved for estimating the news-augmented models in the next step, so all models are compared on the same days.

**Why**
These are the standard benchmarks in volatility forecasting. A news-based model is only useful if it beats them out of sample.

**Results**
- All models converged for every stock and month; 28,482 forecasts per model in the main evaluation period and 11,000 in the robustness period.
- Main sample (2018 – May 2020): GJR-GARCH-t has the lowest loss on both measures (MSE 373.2, QLIKE 2.185), followed by GARCH-t, GARCH-N and EWMA (MSE 396.3, QLIKE 2.280).
- Robustness sample (2023): the GARCH-type models perform almost identically (QLIKE 1.997–2.002); EWMA is again the weakest (QLIKE 2.034).
- Allowing for fat tails and asymmetric reactions to negative returns improves forecasts mainly in the turbulent main period, which includes the COVID-19 crash.
- Saved as `forecasts_baseline.csv.gz`.

In [ ]:
!pip install -q arch

from google.colab import drive
drive.mount("/content/drive")
import os, warnings, numpy as np, pandas as pd
os.chdir("/content/drive/MyDrive/Thesis_new")
from arch import arch_model
from joblib import Parallel, delayed

returns = pd.read_csv("returns_daily.csv", index_col=0, parse_dates=True)
main = pd.read_csv("sample_main.csv")["ticker"].tolist()
robust = pd.read_csv("sample_robust.csv")["ticker"].tolist()

SPECS = {"GARCH-N": dict(vol="GARCH", p=1, q=1, dist="normal"),
         "GARCH-t": dict(vol="GARCH", p=1, q=1, dist="t"),
         "GJR-t":   dict(vol="GARCH", p=1, o=1, q=1, dist="t")}

def forecast_stock(ticker, start, end, lam=0.94):
    warnings.filterwarnings("ignore")
    r = returns[ticker].dropna()
    rows = []

    # EWMA: the variance forecast for day i uses returns up to day i-1
    r2 = (r ** 2).to_numpy()
    s = np.empty(len(r)); s[0] = r2[:250].mean()
    for i in range(1, len(r)):
        s[i] = lam * s[i - 1] + (1 - lam) * r2[i - 1]
    ewma = pd.Series(s, index=r.index)[start:end]
    rows.append(pd.DataFrame({"date": ewma.index, "model": "EWMA", "forecast": ewma.values}))

    # GARCH-type models: re-estimated monthly on all earlier data, then fixed for that month
    n_fail = 0
    for m in pd.period_range(start, end, freq="M"):
        m_start, m_end = m.start_time, m.end_time
        est = r[r.index < m_start]
        if len(est) < 750:                       # require about 3 years of history
            continue
        r_upto = r[r.index <= m_end]
        origin = est.index[-1]                   # last trading day before the month
        for name, spec in SPECS.items():
            try:
                res = arch_model(est, mean="Zero", **spec).fit(disp="off")
                if res.convergence_flag != 0:
                    n_fail += 1
                fixed = arch_model(r_upto, mean="Zero", **spec).fix(res.params)
                f = fixed.forecast(horizon=1, start=origin, reindex=False).variance["h.1"]
                pos = r.index.get_indexer(f.index) + 1   # forecast made on day t is for day t+1
                keep = pos < len(r)
                f = pd.Series(f.values[keep], index=r.index[pos[keep]])
                f = f[(f.index >= m_start) & (f.index <= m_end)]
                rows.append(pd.DataFrame({"date": f.index, "model": name, "forecast": f.values}))
            except Exception:
                n_fail += 1
    out = pd.concat(rows, ignore_index=True)
    out["ticker"] = ticker
    return out, n_fail

# Run for both samples (in parallel across stocks)
results = []
for name, stocks, start, end in [("main", main, "2017-01-01", "2020-05-31"),
                                 ("robust", robust, "2022-01-01", "2023-12-31")]:
    out = Parallel(n_jobs=-1)(delayed(forecast_stock)(t, start, end) for t in stocks)
    df = pd.concat([o for o, _ in out]); df["sample"] = name
    results.append(df)
    print(f"{name}: {len(stocks)} stocks done | non-converged fits: {sum(nf for _, nf in out)}")
fc = pd.concat(results, ignore_index=True)

# Attach the realized squared return and compute losses
r2 = (returns ** 2).stack().rename("r2").rename_axis(["date", "ticker"]).reset_index()
fc = fc.merge(r2, on=["date", "ticker"], how="left")
fc["eval"] = fc["date"] >= pd.to_datetime(fc["sample"].map({"main": "2018-01-01", "robust": "2023-01-01"}))
fc["MSE"] = (fc["r2"] - fc["forecast"]) ** 2
fc["QLIKE"] = np.log(fc["forecast"]) + fc["r2"] / fc["forecast"]

ev = fc[fc["eval"]]
print("\nAverage loss in the evaluation period (lower is better):")
print(ev.groupby(["sample", "model"])[["MSE", "QLIKE"]].mean().round(3))
print("\nForecasts per model:\n", ev.groupby(["sample", "model"]).size())
print("Missing forecasts:", fc["forecast"].isna().sum())

fc.to_csv("forecasts_baseline.csv.gz", index=False)
print("\nSaved forecasts_baseline.csv.gz")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
main: 47 stocks done | non-converged fits: 0
robust: 44 stocks done | non-converged fits: 0

Average loss in the evaluation period (lower is better):
                    MSE  QLIKE
sample model                  
main   EWMA     396.277  2.280
       GARCH-N  385.319  2.223
       GARCH-t  384.123  2.212
       GJR-t    373.215  2.185
robust EWMA     117.304  2.034
       GARCH-N  116.966  1.998
       GARCH-t  118.123  2.002
       GJR-t    117.249  1.997

Forecasts per model:
 sample  model  
main    EWMA       28482
        GARCH-N    28482
        GARCH-t    28482
        GJR-t      28482
robust  EWMA       11000
        GARCH-N    11000
        GARCH-t    11000
        GJR-t      11000
dtype: int64
Missing forecasts: 0

Saved forecasts_baseline.csv.gz


## Step 8: News-augmented volatility forecasts

**What this cell does**
1. Extends each baseline forecast with the daily news features:
   σ²_news(t) = σ²_baseline(t) × exp(c + b₁·log(1 + n_news) + b₂·neg_share + b₃·sent)
   The news term scales the baseline forecast up or down; the exponential keeps forecasts positive.
2. Estimates the coefficients by quasi-maximum likelihood (minimising the QLIKE loss), pooled across all stocks, on an expanding window that starts at the beginning of the sample window (January 2017 / January 2022) and is re-estimated every month using only past data.
3. Compares each news model with
   - the untouched baseline forecast, and
   - a *recalibrated baseline* estimated in exactly the same way but with the constant c only, so any difference comes from the news features alone.
4. Tests differences in forecast accuracy with the Diebold–Mariano test (Diebold and Mariano, 1995): the daily QLIKE difference is averaged across stocks, with Newey–West standard errors (5 lags).
5. Reports in-sample coefficients for the whole window, with standard errors clustered by date.

**Robustness checks**
- All headlines instead of the relevance-filtered ones
- News lagged by one extra trading day
- Ordinary least squares instead of quasi-maximum likelihood (saved in the results file)
- 2023 sample in addition to the main sample

**Results**

*In-sample (whole window, GJR-GARCH-t baseline, standard errors clustered by date):*
- Main sample: news intensity strongly predicts next-day variance (coefficient 0.434, t = 12.9). A day with one relevant headline instead of none raises expected variance by about 35%. More negative tone also raises volatility (sentiment coefficient −0.413, t = −6.6): a shift in average sentiment from 0 to −0.5 raises expected variance by about 23%.
- Robustness sample (2023): news intensity remains significant (0.267, t = 6.2; about +20% per headline), but tone no longer adds significant information.
- `neg_share` has a negative coefficient because it overlaps strongly with `sent` (multicollinearity); the two are interpreted jointly as news tone.

*Out-of-sample (QLIKE, Diebold–Mariano tests):*
- With the GJR-GARCH-t baseline, adding news intensity and tone lowers QLIKE from 2.185 to 2.170 in the main sample (−0.7%) and from 1.997 to 1.986 in 2023 (−0.6%).
- Against a recalibrated baseline estimated in the same way without news, these improvements are significant (p = 0.035 main; p = 0.038 robustness). Against the untouched baseline they are only marginal (p = 0.085 main; p = 0.29 robustness).
- For the weaker EWMA baseline, news improves forecasts significantly in both samples (p < 0.05).
- With an additional one-day lag, the improvement disappears in all models: the predictive content of news is short-lived.
- Unfiltered headlines perform as well as or slightly better than relevance-filtered ones, suggesting that the overall amount of coverage measures investor attention.
- Under MSE, news models perform worse, because MSE is dominated by a few extreme days (e.g. March 2020) on which higher forecasts are penalised heavily.

*Robustness (OLS estimation):* with ordinary least squares instead of quasi-maximum likelihood, news does not improve forecasts in the main sample; in 2023, only the unfiltered-headline variant is significant against the recalibrated baseline (p = 0.006 for GJR-t). The results therefore depend on the estimation method, with the QML results being the more reliable because QML optimises the evaluation loss and is less sensitive to extreme observations.

- Saved as `results_news_models.csv` and `forecasts_news.csv.gz`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, numpy as np, pandas as pd, statsmodels.api as sm
from scipy.optimize import minimize
os.chdir("/content/drive/MyDrive/Thesis_new")

fc = pd.read_csv("forecasts_baseline.csv.gz", parse_dates=["date"])
feat = pd.read_csv("features_daily.csv.gz", parse_dates=["date"]).sort_values(["sample", "ticker", "date"])

# 1. Model inputs
for sfx in ["", "_all"]:
    feat[f"log_news{sfx}"] = np.log1p(feat[f"n_news{sfx}"])
news_cols = ["log_news", "neg_share", "sent"]
lag_cols = [c + "_lag2" for c in news_cols]
feat[lag_cols] = feat.groupby(["sample", "ticker"])[news_cols].shift(1).to_numpy()
feat = feat[feat["in_window"]]
df = fc.merge(feat, on=["sample", "ticker", "date"], how="inner")

VARIANTS = {"News intensity":             ["log_news"],
            "News intensity + tone":      news_cols,
            "All headlines (unfiltered)": ["log_news_all", "neg_share_all", "sent_all"],
            "Extra one-day lag":          lag_cols}
WIN_START = {"main": "2017-01-01", "robust": "2022-01-01"}
EVAL_START = {"main": "2018-01-01", "robust": "2023-01-01"}

# 2. Estimators for the news adjustment: forecast = baseline forecast x g(news)
def fit_qml(h, r2, X):
    """Quasi-maximum likelihood: g = exp(X b), chosen to minimise the QLIKE loss."""
    f = lambda b: np.mean(X @ b + r2 / (h * np.exp(X @ b)))
    grad = lambda b: X.T @ (1 - r2 / (h * np.exp(X @ b))) / len(h)
    return minimize(f, np.zeros(X.shape[1]), jac=grad, method="BFGS").x

def fit_ols(h, r2, X):
    """Robustness: g = X b by least squares on r2 / h (clipped below at 0.05 when forecasting)."""
    return np.linalg.lstsq(X, r2 / h, rcond=None)[0]

def adjust(X, b, method):
    return np.exp(X @ b) if method == "QML" else np.clip(X @ b, 0.05, None)

# 3. Expanding-window estimation, re-estimated monthly, pooled across stocks
rows = []
for method, fit in [("QML", fit_qml), ("OLS", fit_ols)]:
    for (sample, model), g in df.groupby(["sample", "model"]):
        ev = g[g["date"] >= EVAL_START[sample]]
        out = {"Baseline": ev["forecast"].to_numpy().copy()}
        for name in ["Recalibrated baseline"] + list(VARIANTS):
            out[name] = np.full(len(ev), np.nan)
        for m in pd.period_range(EVAL_START[sample], ev["date"].max(), freq="M"):
            tr = g[(g["date"] >= WIN_START[sample]) & (g["date"] < m.start_time)]
            te = ((ev["date"] >= m.start_time) & (ev["date"] <= m.end_time)).to_numpy()
            h_tr, r2_tr, h_te = tr["forecast"].to_numpy(), tr["r2"].to_numpy(), ev["forecast"].to_numpy()[te]
            one_tr, one_te = np.ones((len(tr), 1)), np.ones((te.sum(), 1))
            b = fit(h_tr, r2_tr, one_tr)
            out["Recalibrated baseline"][te] = h_te * adjust(one_te, b, method)
            for name, cols in VARIANTS.items():
                X_tr = np.hstack([one_tr, tr[cols].to_numpy()])
                X_te = np.hstack([one_te, ev.loc[te, cols].to_numpy()])
                b = fit(h_tr, r2_tr, X_tr)
                out[name][te] = h_te * adjust(X_te, b, method)
        for name, f in out.items():
            rows.append(pd.DataFrame({"method": method, "sample": sample, "base_model": model, "variant": name,
                                      "date": ev["date"].to_numpy(), "ticker": ev["ticker"].to_numpy(),
                                      "r2": ev["r2"].to_numpy(), "forecast": f}))
res = pd.concat(rows, ignore_index=True)
res["MSE"] = (res["r2"] - res["forecast"]) ** 2
res["QLIKE"] = np.log(res["forecast"]) + res["r2"] / res["forecast"]

# 4. Diebold-Mariano tests on QLIKE: daily average loss difference across stocks, Newey-West (5 lags)
def dm_test(a, b):
    d = a.groupby("date")["QLIKE"].mean() - b.groupby("date")["QLIKE"].mean()
    fit = sm.OLS(d.to_numpy(), np.ones(len(d))).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    return fit.tvalues[0], fit.pvalues[0]

table = []
for (method, sample, model, variant), g in res.groupby(["method", "sample", "base_model", "variant"], sort=False):
    key = (res["method"] == method) & (res["sample"] == sample) & (res["base_model"] == model)
    row = {"method": method, "sample": sample, "base_model": model, "variant": variant,
           "MSE": g["MSE"].mean(), "QLIKE": g["QLIKE"].mean()}
    if variant != "Baseline":
        row["DM t (vs baseline)"], row["p (vs baseline)"] = dm_test(g, res[key & (res["variant"] == "Baseline")])
    if variant not in ("Baseline", "Recalibrated baseline"):
        row["DM t (vs recalibrated)"], row["p (vs recalibrated)"] = dm_test(g, res[key & (res["variant"] == "Recalibrated baseline")])
    table.append(row)
table = pd.DataFrame(table)
pd.set_option("display.width", 250, "display.max_rows", 200)
print("Out-of-sample results, main method (QML); negative DM t = news model is more accurate:")
print(table[table["method"] == "QML"].drop(columns="method").round(3).to_string(index=False))

# 5. In-sample coefficients (whole window, GJR-t baseline), standard errors clustered by date
def qml_coefficients(g, cols):
    h, r2 = g["forecast"].to_numpy(), g["r2"].to_numpy()
    X = np.hstack([np.ones((len(g), 1)), g[cols].to_numpy()])
    b = fit_qml(h, r2, X)
    u = r2 / (h * np.exp(X @ b))
    H = (X * u[:, None]).T @ X / len(h)
    S = pd.DataFrame(X * (1 - u)[:, None]).groupby(g["date"].to_numpy()).sum().to_numpy()
    V = np.linalg.inv(H) @ (S.T @ S / len(h) ** 2) @ np.linalg.inv(H)
    se = np.sqrt(np.diag(V))
    return pd.DataFrame({"coef": b, "std. err.": se, "t": b / se}, index=["const"] + cols)

for sample in ["main", "robust"]:
    g = df[(df["sample"] == sample) & (df["model"] == "GJR-t")]
    print(f"\nIn-sample QML coefficients, {sample} sample (GJR-t baseline):")
    print(qml_coefficients(g, news_cols).round(3))

table.round(4).to_csv("results_news_models.csv", index=False)
res.to_csv("forecasts_news.csv.gz", index=False)
print("\nSaved results_news_models.csv and forecasts_news.csv.gz")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Out-of-sample results, main method (QML); negative DM t = news model is more accurate:
sample base_model                    variant     MSE  QLIKE  DM t (vs baseline)  p (vs baseline)  DM t (vs recalibrated)  p (vs recalibrated)
  main       EWMA                   Baseline 396.277  2.280                 NaN              NaN                     NaN                  NaN
  main       EWMA      Recalibrated baseline 410.248  2.254              -1.597            0.110                     NaN                  NaN
  main       EWMA             News intensity 426.168  2.233              -2.788            0.005                  -2.294                0.022
  main       EWMA      News intensity + tone 427.517  2.228              -3.040            0.002                  -2.711                0.007
  main       EWMA All headlines (unfiltered) 426.332  2.232              -

## Step 9: Tables and figures

**What this cell does**
Creates the tables and figures used in the thesis and saves each table as CSV and LaTeX (booktabs) and each figure as PDF and PNG in the folder `thesis_outputs`:
- Table 1: descriptive statistics of daily returns (main sample)
- Table 2: summary of the news data (headline counts, relevance filter, FinBERT label shares)
- Table 3: out-of-sample accuracy of the baseline models
- Table 4: out-of-sample accuracy of the news-augmented models (GJR-GARCH-t baseline, main result)
- Table 5: in-sample coefficients of the news adjustment
- Appendix tables: news-augmented models for every baseline, QML and OLS
- Figure 1: news coverage over time
- Figure 2: squared returns relative to the GJR-GARCH-t forecast, by number of relevant headlines on the previous day
- Figure 3: cumulative QLIKE difference between the news model and the baselines over time
- Figure 4: example stock (Gilead Sciences) during the COVID-19 crash

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.dates as mdates
from scipy.optimize import minimize
os.chdir("/content/drive/MyDrive/Thesis_new")
os.makedirs("thesis_outputs", exist_ok=True)
OUT = "thesis_outputs/"

# Figure style for print: thin lines, light grid, fixed colours
BLUE, ORANGE, GREY, INK = "#2a78d6", "#eb6834", "#8a8984", "#0b0b0b"
plt.rcParams.update({"font.family": "serif", "font.size": 10, "axes.edgecolor": "#c9c8c3",
                     "axes.labelcolor": INK, "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "axes.grid": True, "grid.color": "#e8e7e3", "grid.linewidth": 0.6,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "legend.frameon": False, "figure.dpi": 150, "savefig.bbox": "tight"})

def save_table(df, name, fmt="{:.3f}"):
    """Save a table as CSV and as a LaTeX tabular (caption and label are set in main.tex)."""
    df.to_csv(OUT + name + ".csv")
    esc = lambda x: str(x).replace("%", "\\%").replace("&", "\\&")
    df = df.copy()
    df.index = [esc(i) for i in df.index]
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = pd.MultiIndex.from_tuples([tuple(esc(c) for c in col) for col in df.columns])
    else:
        df.columns = [esc(c) for c in df.columns]
    latex = df.style.format(fmt, na_rep="--").to_latex(hrules=True, multicol_align="c",
                                                       column_format="l" + "r" * df.shape[1])
    open(OUT + name + ".tex", "w").write(latex)

def save_fig(fig, name):
    fig.savefig(OUT + name + ".pdf"); fig.savefig(OUT + name + ".png", dpi=300); plt.close(fig)

feat = pd.read_csv("features_daily.csv.gz", parse_dates=["date"])
feat = feat[feat["in_window"]]
fc = pd.read_csv("forecasts_baseline.csv.gz", parse_dates=["date"])
res = pd.read_csv("forecasts_news.csv.gz", parse_dates=["date"])
table = pd.read_csv("results_news_models.csv")
desc = pd.read_csv("descriptive_stats_main.csv", index_col=0)
news = pd.read_csv("news_scored.csv.gz", parse_dates=["Date"])
main = pd.read_csv("sample_main.csv")["ticker"].tolist()
robust = pd.read_csv("sample_robust.csv")["ticker"].tolist()
WINDOWS = {"main": ("2017-01-01", "2020-05-31", main), "robust": ("2022-01-01", "2023-12-31", robust)}
SAMPLE_NAMES = {"main": "Main (2017 - May 2020)", "robust": "Robustness (2022 - 2023)"}
EVAL_NAMES = {"main": "Main (evaluation 2018 - May 2020)", "robust": "Robustness (evaluation 2023)"}
SHORT = {"main": "Main sample", "robust": "Robustness sample"}

# ---------- Table 1: descriptive statistics (summary across the 47 stocks) ----------
t1 = desc.drop(columns="Observations").agg(["mean", "median", "min", "max"]).T
t1.columns = ["Mean", "Median", "Min", "Max"]
save_table(t1, "table1_descriptive_stats", "{:.2f}")

# ---------- Table 2: news data summary ----------
rows = {}
for s, (a, b, stocks) in WINDOWS.items():
    w = feat[feat["sample"] == s]
    d = news["Date"].dt.tz_convert(None)
    n = news[news["Stock_symbol"].isin(stocks) & (d >= a) & (d <= b)]
    rows[SAMPLE_NAMES[s]] = {
        "Stocks": w["ticker"].nunique(), "Trading days": w["date"].nunique(),
        "Headlines (all)": int(w["n_news_all"].sum()), "Relevant headlines": int(w["n_news"].sum()),
        "Relevant share (%)": 100 * w["n_news"].sum() / w["n_news_all"].sum(),
        "Relevant headlines per stock-day": w["n_news"].mean(),
        "Stock-days with relevant news (%)": 100 * (w["n_news"] > 0).mean(),
        "FinBERT positive (%)": 100 * (n["label"] == "positive").mean(),
        "FinBERT neutral (%)": 100 * (n["label"] == "neutral").mean(),
        "FinBERT negative (%)": 100 * (n["label"] == "negative").mean()}
t2 = pd.DataFrame(rows)
t2 = t2.apply(lambda col: [f"{v:,.0f}" if k in ("Stocks", "Trading days", "Headlines (all)", "Relevant headlines")
                           else f"{v:.2f}" for k, v in col.items()])
save_table(t2, "table2_news_summary", "{}")

# ---------- Table 3: baseline forecast accuracy ----------
base = table[(table["method"] == "QML") & (table["variant"] == "Baseline")]
t3 = base.pivot(index="base_model", columns="sample", values=["QLIKE", "MSE"])
t3 = t3.reindex(["EWMA", "GARCH-N", "GARCH-t", "GJR-t"])
t3.columns = [f"{m} ({SAMPLE_NAMES[s].split(' ')[0]})" for m, s in t3.columns]
save_table(t3, "table3_baseline_accuracy")

# ---------- Table 4: news models, GJR-GARCH-t baseline (main result) + appendix tables ----------
def news_table(method, model):
    t = table[(table["method"] == method) & (table["base_model"] == model)]
    out = []
    for s in ["main", "robust"]:
        x = t[t["sample"] == s].set_index("variant")[["QLIKE", "p (vs baseline)", "p (vs recalibrated)"]]
        x.columns = pd.MultiIndex.from_product([[SHORT[s]], ["QLIKE", "p (baseline)", "p (recalibrated)"]])
        out.append(x)
    out = pd.concat(out, axis=1)
    out.index.name = None
    return out
save_table(news_table("QML", "GJR-t"), "table4_news_models_gjr")
for m in ["EWMA", "GARCH-N", "GARCH-t", "GJR-t"]:
    save_table(news_table("QML", m), f"tableA_news_models_{m}_QML")
    save_table(news_table("OLS", m), f"tableA_news_models_{m}_OLS")

# ---------- Table 5: in-sample coefficients of the news adjustment (QML, GJR-t baseline) ----------
def fit_qml(h, r2, X):
    f = lambda b: np.mean(X @ b + r2 / (h * np.exp(X @ b)))
    grad = lambda b: X.T @ (1 - r2 / (h * np.exp(X @ b))) / len(h)
    return minimize(f, np.zeros(X.shape[1]), jac=grad, method="BFGS").x

def qml_coefficients(g, cols):
    h, r2 = g["forecast"].to_numpy(), g["r2"].to_numpy()
    X = np.hstack([np.ones((len(g), 1)), g[cols].to_numpy()])
    b = fit_qml(h, r2, X)
    u = r2 / (h * np.exp(X @ b))
    H = (X * u[:, None]).T @ X / len(h)
    S = pd.DataFrame(X * (1 - u)[:, None]).groupby(g["date"].to_numpy()).sum().to_numpy()
    V = np.linalg.inv(H) @ (S.T @ S / len(h) ** 2) @ np.linalg.inv(H)
    return b, np.sqrt(np.diag(V))

feat["log_news"] = np.log1p(feat["n_news"])
cols = ["log_news", "neg_share", "sent"]
t5 = {}
for s in ["main", "robust"]:
    g = fc[(fc["model"] == "GJR-t") & (fc["sample"] == s)].merge(feat, on=["sample", "ticker", "date"])
    coef, se = qml_coefficients(g, cols)
    t5[(SHORT[s], "Coef.")] = coef
    t5[(SHORT[s], "Std. err.")] = se
    t5[(SHORT[s], "t-stat.")] = coef / se
t5 = pd.DataFrame(t5, index=["Constant", "log(1 + headlines)", "Share of negative headlines", "Average sentiment"])
save_table(t5, "table5_news_coefficients")

# ---------- Figure 1: news coverage over time ----------
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True, gridspec_kw={"width_ratios": [41, 24]})
for ax, s in zip(axes, ["main", "robust"]):
    w = feat[feat["sample"] == s]
    m = w.groupby(w["date"].dt.to_period("M"))[["n_news_all", "n_news"]].sum() / w["ticker"].nunique()
    x = m.index.to_timestamp()
    ax.plot(x, m["n_news_all"], color=BLUE, lw=1.6, label="All headlines")
    ax.plot(x, m["n_news"], color=ORANGE, lw=1.6, label="Relevant headlines")
    ax.set_title(SAMPLE_NAMES[s], fontsize=10, loc="left")
    if s == "main":
        ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    else:
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7])); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
top = max((feat[feat["sample"] == s].groupby(feat["date"].dt.to_period("M"))["n_news_all"].sum()
           / feat.loc[feat["sample"] == s, "ticker"].nunique()).max() for s in ["main", "robust"])
axes[0].set_ylim(0, top * 1.08)
axes[0].set_ylabel("Headlines per stock and month")
axes[0].legend(loc="upper left")
save_fig(fig, "figure1_news_coverage")

# ---------- Figure 2: squared return relative to the GJR forecast, by news volume ----------
g = fc[(fc["model"] == "GJR-t") & (fc["sample"] == "main")].merge(feat, on=["sample", "ticker", "date"])
g["ratio"] = g["r2"] / g["forecast"]
g["bucket"] = pd.cut(g["n_news"], [-1, 0, 1, 3, np.inf], labels=["0", "1", "2-3", "4+"])
b = g.groupby("bucket", observed=True)["ratio"].agg(["mean", "count"])
fig, ax = plt.subplots(figsize=(4.8, 3.2))
labels = [f"{k}\n(n = {n:,})" for k, n in zip(b.index.astype(str), b["count"])]
bars = ax.bar(labels, b["mean"], color=BLUE, width=0.6, zorder=2)
ax.axhline(1, color=GREY, lw=1, ls="--", zorder=3)
for rect, v in zip(bars, b["mean"]):
    ax.text(rect.get_x() + rect.get_width() / 2, v - 0.06, f"{v:.2f}", ha="center", va="top", fontsize=9, color="white", zorder=4)
ax.set_xlabel("Relevant headlines on the previous day")
ax.set_ylabel("Mean of $r_t^2 \\,/\\, \\hat\\sigma^2_{GJR,t}$")
ax.set_ylim(0, b["mean"].max() * 1.1)
ax.grid(axis="x", visible=False)
save_fig(fig, "figure2_news_volume_vs_volatility")

# ---------- Figure 3: cumulative QLIKE difference of the news model over time ----------
def daily(variant, s, method="QML", model="GJR-t"):
    x = res[(res["method"] == method) & (res["sample"] == s) & (res["base_model"] == model) & (res["variant"] == variant)]
    return x.groupby("date")["QLIKE"].mean()
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), gridspec_kw={"width_ratios": [29, 12]})
for ax, s in zip(axes, ["main", "robust"]):
    news_l = daily("News intensity + tone", s)
    d1 = (news_l - daily("Baseline", s)).cumsum()
    d2 = (news_l - daily("Recalibrated baseline", s)).cumsum()
    ax.plot(d1.index, d1, color=BLUE, lw=1.6, label="vs. baseline")
    ax.plot(d2.index, d2, color=ORANGE, lw=1.6, label="vs. recalibrated baseline")
    ax.axhline(0, color=GREY, lw=1)
    if s == "main":
        ax.axvspan(pd.Timestamp("2020-02-20"), pd.Timestamp("2020-04-30"), color="#e8e7e3", lw=0, zorder=0)
        ax.text(pd.Timestamp("2020-02-12"), ax.get_ylim()[1] * 0.9, "COVID-19\ncrash", fontsize=8, color="#52514e", va="top", ha="right")
        ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    else:
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10])); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    ax.set_title(EVAL_NAMES[s], fontsize=10, loc="left")
axes[0].set_ylabel("Cumulative QLIKE difference\n(below 0 = news model better)")
axes[0].legend(loc="lower left")
save_fig(fig, "figure3_cumulative_qlike_gain")

# ---------- Figure 4: example stock during the COVID-19 crash ----------
ex = "GILD"
x = res[(res["method"] == "QML") & (res["sample"] == "main") & (res["base_model"] == "GJR-t") &
        (res["ticker"] == ex) & (res["date"] >= "2020-01-15")]
piv = x.pivot(index="date", columns="variant", values="forecast")
r2 = x.drop_duplicates("date").set_index("date")["r2"]
ann = lambda v: np.sqrt(v * 252)          # daily variance in %^2 -> annualised volatility in %
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.scatter(r2.index, ann(r2), s=9, color=GREY, label="Realised proxy $\\sqrt{252\\,r_t^2}$", zorder=2)
ax.plot(piv.index, ann(piv["Baseline"]), color=BLUE, lw=1.6, label="GJR-GARCH-t", zorder=3)
ax.plot(piv.index, ann(piv["News intensity + tone"]), color=ORANGE, lw=1.6, label="GJR-GARCH-t + news", zorder=3)
ax.set_ylabel("Annualised volatility (%)")
ax.set_title(f"{ex} (Gilead Sciences), January–May 2020", fontsize=10, loc="left")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.legend(loc="upper left")
save_fig(fig, "figure4_example_gild")

print("Created files:"); print("\n".join(sorted(os.listdir(OUT))))
shutil.make_archive("thesis_outputs", "zip", "thesis_outputs")
print("\nAll tables and figures zipped as thesis_outputs.zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Created files:
figure1_news_coverage.pdf
figure1_news_coverage.png
figure2_news_volume_vs_volatility.pdf
figure2_news_volume_vs_volatility.png
figure3_cumulative_qlike_gain.pdf
figure3_cumulative_qlike_gain.png
figure4_example_gild.pdf
figure4_example_gild.png
table1_descriptive_stats.csv
table1_descriptive_stats.tex
table2_news_summary.csv
table2_news_summary.tex
table3_baseline_accuracy.csv
table3_baseline_accuracy.tex
table4_news_models_gjr.csv
table4_news_models_gjr.tex
table5_news_coefficients.csv
table5_news_coefficients.tex
tableA_news_models_EWMA_OLS.csv
tableA_news_models_EWMA_OLS.tex
tableA_news_models_EWMA_QML.csv
tableA_news_models_EWMA_QML.tex
tableA_news_models_GARCH-N_OLS.csv
tableA_news_models_GARCH-N_OLS.tex
tableA_news_models_GARCH-N_QML.csv
tableA_news_models_GARCH-N_QML.tex
tableA_news_models_GARCH-t_OLS.csv
tableA_news_models_GARCH-t_OLS.

## Step 10: Validation of the relevance filter

**What this cell does**
1. Draws a random sample of 100 non-roundup headlines from the main sample window (fixed random seed 2024, so the sample can be reproduced).
2. Compares the automatic relevance filter from Step 6 with reference labels stored in `relevance_validation_labels.csv`. A headline is labelled as relevant (1) if it names or clearly refers to the company it is tagged to.
3. Reports agreement, precision, recall and Cohen's kappa.

**Reference labels**
The 100 headlines were labelled according to the rule above and checked by the author; eight borderline cases (e.g. company named only within a list, or only by nickname or one-letter ticker) were decided explicitly.


**Results**
- Agreement in 98 of 100 headlines; precision 1.00, recall 0.95, Cohen's kappa 0.96.
- The filter missed two relevant headlines: one refers to Caterpillar by its nickname "Cat", and one lists Citigroup only by its one-letter ticker "C".
- The filter is therefore conservative: it almost never keeps off-topic headlines and only rarely drops relevant ones.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, re, numpy as np, pandas as pd
os.chdir("/content/drive/MyDrive/Thesis_new")

# 1. Draw the validation sample (fixed random seed, so the same 100 headlines are drawn every time)
news = pd.read_csv("news_scored.csv.gz", parse_dates=["Date"])
main = pd.read_csv("sample_main.csv")["ticker"].tolist()
d = news["Date"].dt.tz_convert(None)
pool = news[news["Stock_symbol"].isin(main) & (d >= "2017-01-01") & (d <= "2020-05-31") & ~news["roundup"]]
pool = pool.drop_duplicates(["Stock_symbol", "Article_title"])
sample = pool.sample(100, random_state=2024)[["Stock_symbol", "Article_title"]].reset_index(drop=True)

# 2. Attach the reference labels (1 = headline names or clearly refers to the tagged company)
labels = pd.read_csv("relevance_validation_labels.csv")
sample = sample.merge(labels[["Stock_symbol", "Article_title", "about_company"]],
                      on=["Stock_symbol", "Article_title"], how="left")
assert sample["about_company"].notna().all(), "Sample does not match the label file"

# 3. Apply the same relevance filter as in Step 6
NAMES = {"ABBV": ["AbbVie"], "ABT": ["Abbott"], "ACN": ["Accenture"], "ADI": ["Analog Devices"],
         "AMAT": ["Applied Materials"], "AMD": ["AMD", "Advanced Micro"], "AMGN": ["Amgen"], "ANET": ["Arista"],
         "APH": ["Amphenol"], "AVGO": ["Broadcom"], "AXP": ["American Express", "Amex"],
         "BAC": ["Bank of America", "BofA"], "BLK": ["BlackRock"], "C": ["Citigroup", "Citi"],
         "CAT": ["Caterpillar"], "COP": ["ConocoPhillips", "Conoco"], "COST": ["Costco"], "CRM": ["Salesforce"],
         "CVX": ["Chevron"], "DE": ["Deere"], "DHR": ["Danaher"], "GE": ["General Electric", "GE"],
         "GILD": ["Gilead"], "GS": ["Goldman"], "HD": ["Home Depot"],
         "JNJ": ["Johnson & Johnson", "Johnson and Johnson", "J&J"], "KO": ["Coca-Cola", "Coca Cola", "Coke"],
         "MRK": ["Merck"], "MS": ["Morgan Stanley"], "MU": ["Micron"], "NEE": ["NextEra"], "NFLX": ["Netflix"],
         "NVDA": ["Nvidia"], "ORCL": ["Oracle"], "PANW": ["Palo Alto Networks"], "PEP": ["PepsiCo", "Pepsi"],
         "PFE": ["Pfizer"], "PM": ["Philip Morris"], "QCOM": ["Qualcomm"], "T": ["AT&T"],
         "TJX": ["TJX", "T.J. Maxx", "TJ Maxx"], "TMO": ["Thermo Fisher"], "TMUS": ["T-Mobile"],
         "TXN": ["Texas Instruments"], "UNP": ["Union Pacific"], "V": ["Visa"], "WFC": ["Wells Fargo"]}

def is_relevant(title, ticker):
    if re.search(r"\(" + re.escape(ticker) + r"\)", title):
        return True
    if len(ticker) >= 3 and re.search(r"\b" + re.escape(ticker) + r"\b", title):
        return True
    for name in NAMES[ticker]:
        flags = 0 if name.isupper() else re.IGNORECASE
        if re.search(r"(?<![A-Za-z])" + re.escape(name) + r"(?![A-Za-z])", title, flags):
            return True
    return False

sample["filter"] = [int(is_relevant(t, s)) for t, s in zip(sample["Article_title"], sample["Stock_symbol"])]

# 4. Agreement statistics
y, f = sample["about_company"].astype(int), sample["filter"]
tp, fp = int(((f == 1) & (y == 1)).sum()), int(((f == 1) & (y == 0)).sum())
fn, tn = int(((f == 0) & (y == 1)).sum()), int(((f == 0) & (y == 0)).sum())
agree = (tp + tn) / len(sample)
p_exp = y.mean() * f.mean() + (1 - y.mean()) * (1 - f.mean())
kappa = (agree - p_exp) / (1 - p_exp)
print(f"Agreement: {tp + tn} of {len(sample)} headlines ({agree:.0%})")
print(f"Precision: {tp / (tp + fp):.2f}  (share of headlines kept by the filter that are about the company)")
print(f"Recall:    {tp / (tp + fn):.2f}  (share of company headlines that the filter keeps)")
print(f"Cohen's kappa: {kappa:.2f}")
print(f"\nConfusion matrix: TP={tp}, FP={fp}, FN={fn}, TN={tn}")
pd.set_option("display.max_colwidth", 110, "display.width", 200)
print("\nDisagreements:\n", sample[sample["filter"] != y][["Stock_symbol", "Article_title", "about_company", "filter"]])

sample.to_csv("relevance_validation_results.csv", index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Agreement: 98 of 100 headlines (98%)
Precision: 1.00  (share of headlines kept by the filter that are about the company)
Recall:    0.95  (share of company headlines that the filter keeps)
Cohen's kappa: 0.96

Confusion matrix: TP=40, FP=0, FN=2, TN=58

Disagreements:
    Stock_symbol                                                                                              Article_title  about_company  filter
91          CAT  From Cat Q3 Earnings Conference Call: Co. Committed To High Single Digit Dividend Increase Over The Years              1       0
95            C                                                      Financial Sector Update for 03/26/2019: MARA,FDS,BK,C              1       0
